In [1]:
from pyspark.sql import SparkSession
import getpass
username =getpass.getuser()
spark =SparkSession.builder.config('spark.ui.port','0') \
.config('spark.shuffle.useOldFetchProtocol','true') \
.config('spark.sql.warehouse.dir', f'/user/{username}/warehouse') \
.enableHiveSupport() \
.master('yarn') \
.getOrCreate()

In [2]:
from pyspark.sql.functions import col

In [3]:
loan_defaulters_schema = "member_id string, delinq_2yrs float, delinq_amnt float, pub_rec float, pub_rec_bankruptcies float,inq_last_6mths float, total_rec_late_fee float, mths_since_last_delinq float, mths_since_last_record float"

In [4]:
loan_defaulters_df = spark.read \
.format("csv") \
.option("header",True) \
.schema(loan_defaulters_schema) \
.load(f"/user/{username}/lendingclub/raw/loans_defaulter_csv")

In [5]:
loan_defaulters_df

member_id,delinq_2yrs,delinq_amnt,pub_rec,pub_rec_bankruptcies,inq_last_6mths,total_rec_late_fee,mths_since_last_delinq,mths_since_last_record
a4ec00ba67fadf2fe...,1.0,0.0,0.0,0.0,0.0,0.0,9.0,null
4f7a9e6ffaacd5da2...,1.0,0.0,0.0,0.0,0.0,0.0,14.0,null
e935a4c27fc78ae61...,0.0,0.0,0.0,0.0,0.0,0.0,43.0,null
2d32004bd5e1dc3c3...,1.0,0.0,1.0,0.0,2.0,0.0,17.0,58.0
f7116b7f7546a7952...,0.0,0.0,2.0,1.0,0.0,0.0,28.0,55.0
d3aa3a3c95eca5631...,0.0,0.0,0.0,0.0,0.0,68.8,null,null
fc8a2e046eaaba02d...,0.0,0.0,0.0,0.0,0.0,0.0,null,null
577ae670ac2ec7ed3...,0.0,0.0,0.0,0.0,1.0,0.0,null,null
d3792868819649ba9...,0.0,0.0,0.0,0.0,0.0,110.13,null,null
6d3a5a422261348b3...,0.0,0.0,1.0,0.0,1.0,0.0,null,38.0


In [6]:
loan_defaulters_no_null = loan_defaulters_df.withColumn("delinq_2yrs",col("delinq_2yrs").cast("integer")).fillna(0,subset=["delinq_2yrs"])

In [7]:
loan_defaulters_no_null.filter(loan_defaulters_no_null.delinq_2yrs.isNull()).count()

0

In [8]:
loan_defaulters_no_null.groupBy("delinq_2yrs").count().orderBy("count", ascending = False)

delinq_2yrs,count
0,1839141
1,281337
2,81285
3,29545
4,13180
5,6601
6,3719
7,2063
8,1226
9,821


In [9]:
loan_defaulters_no_null.createOrReplaceTempView("loan_defaulters")

In [10]:
loans_def_delinq_df = spark.sql("select member_id,delinq_2yrs, delinq_amnt, int(mths_since_last_delinq) from loan_defaulters where delinq_2yrs > 0 or mths_since_last_delinq > 0")

In [11]:
loans_def_delinq_df

member_id,delinq_2yrs,delinq_amnt,mths_since_last_delinq
a4ec00ba67fadf2fe...,1,0.0,9
4f7a9e6ffaacd5da2...,1,0.0,14
e935a4c27fc78ae61...,0,0.0,43
2d32004bd5e1dc3c3...,1,0.0,17
f7116b7f7546a7952...,0,0.0,28
2a1dd1ae5b20c4dee...,0,0.0,69
c266c3c131861bb8c...,0,0.0,24
199119e026b4bfb14...,1,0.0,21
0a9cef2d26384e853...,1,0.0,15
39ff608d7284a71f5...,1,0.0,21


In [12]:
loans_def_delinq_df.count()

1106163

In [28]:
loans_def_records_enq_df = spark.sql("select member_id,pub_rec,pub_rec_bankruptcies,inq_last_6mths from loan_defaulters where pub_rec > 0.0 or pub_rec_bankruptcies > 0.0 or inq_last_6mths > 0.0")

In [29]:
loans_def_records_enq_df.count()

1070124

In [30]:
loans_def_delinq_df.write \
.option("header", True) \
.format("csv") \
.mode("overwrite") \
.option("path", f"/user/{username}/lendingclub/cleaned/loans_defaulters_deling_csv") \
.save()

In [31]:
loans_def_delinq_df.write \
.format("parquet") \
.mode("overwrite") \
.option("path", f"/user/{username}/lendingclub/cleaned/loans_defaulters_deling_parquet") \
.save()

In [17]:
loans_def_records_enq_df.write \
.option("header", True) \
.format("csv") \
.mode("overwrite") \
.option("path", f"/user/{username}/lendingclub/cleaned/loans_defaulters_records_enq_csv") \
.save()

In [32]:
loans_def_records_enq_df.write \
.format("parquet") \
.mode("overwrite") \
.option("path", f"/user/{username}/lendingclub/cleaned/loans_defaulters_records_enq_parquet") \
.save()

In [33]:
loans_def_records_enq_df

member_id,pub_rec,pub_rec_bankruptcies,inq_last_6mths
2d32004bd5e1dc3c3...,1,0,2
f7116b7f7546a7952...,2,1,0
577ae670ac2ec7ed3...,0,0,1
6d3a5a422261348b3...,1,0,1
7435a5a4b2b825a20...,0,0,3
8d203efd7c1af010c...,1,1,0
c266c3c131861bb8c...,1,0,0
9284503d1d6dcd23b...,1,1,0
c764f6b4666a3a439...,0,0,1
199119e026b4bfb14...,2,0,0


In [35]:
loans_def_records_enq_df.printSchema()

root
 |-- member_id: string (nullable = true)
 |-- pub_rec: integer (nullable = true)
 |-- pub_rec_bankruptcies: integer (nullable = true)
 |-- inq_last_6mths: integer (nullable = true)



In [19]:
loans_def_p_pub_rec_df = loan_defaulters_no_null.withColumn("pub_rec", col("pub_rec").cast("integer")).fillna(0, subset = ["pub_rec"])

In [20]:
loans_def_p_pub_rec_bankruptcies_df = loans_def_p_pub_rec_df.withColumn("pub_rec_bankruptcies", col("pub_rec_bankruptcies").cast("integer")).fillna(0, subset = ["pub_rec_bankruptcies"])

In [21]:
loans_def_p_inq_last_6mths_df = loans_def_p_pub_rec_bankruptcies_df.withColumn("inq_last_6mths", col("inq_last_6mths").cast("integer")).fillna(0, subset = ["inq_last_6mths"])

In [22]:
loans_def_p_inq_last_6mths_df.createOrReplaceTempView("loan_defaulters")

In [23]:
loans_def_detail_records_enq_df = spark.sql("select member_id, pub_rec, pub_rec_bankruptcies, inq_last_6mths from loan_defaulters")

In [24]:
loans_def_detail_records_enq_df

member_id,pub_rec,pub_rec_bankruptcies,inq_last_6mths
a4ec00ba67fadf2fe...,0,0,0
4f7a9e6ffaacd5da2...,0,0,0
e935a4c27fc78ae61...,0,0,0
2d32004bd5e1dc3c3...,1,0,2
f7116b7f7546a7952...,2,1,0
d3aa3a3c95eca5631...,0,0,0
fc8a2e046eaaba02d...,0,0,0
577ae670ac2ec7ed3...,0,0,1
d3792868819649ba9...,0,0,0
6d3a5a422261348b3...,1,0,1


In [25]:
loans_def_detail_records_enq_df.write \
.option("header", True) \
.format("csv") \
.mode("overwrite") \
.option("path", f"/user/{username}/lendingclub/cleaned/loans_def_detail_records_enq_df_csv") \
.save()

In [26]:
loans_def_detail_records_enq_df.write \
.format("parquet") \
.mode("overwrite") \
.option("path", f"/user/{username}/lendingclub/cleaned/loans_def_detail_records_enq_df_parquet") \
.save()